# 01. Staging de Empresas Normalizadas

## Propósito metodológico
Preparar los nombres comerciales del CRM (`leads`) y del registro de proyectos (`horas`) para el carril automatizado de resolución de entidades. Este notebook no asigna RUCs oficiales ni alimenta directamente el modelo final de clustering; su función es construir un universo normalizado que permita evaluar empíricamente qué tan viable es cruzar nombres comerciales contra fuentes oficiales.

En la tesis, esta etapa pertenece al **baseline automatizado**. El modelo final se alimenta posteriormente desde el **Golden Record manual** (`match_final_empresas_verificado.csv`), construido con RUCs verificados.


## Inputs
- `leads.xlsx` (esperado en la raíz del proyecto; fallback: `01_data_ingestion_enrichment/leads.xlsx`)
- `proyectos_empresa.xlsx` (esperado en la raíz del proyecto; fallback: `01_data_ingestion_enrichment/proyectos_empresa.xlsx`)

## Outputs del carril automatizado (se guardan en `02_data_cleaning/outputs/`)
- `leads_companies_clean.csv`
- `horas_empresas_clean.csv`
- `empresas_universe_compilado.csv`

Estos archivos son insumos para el experimento de *Entity Resolution* automático. No reemplazan al diccionario manual verificado que se usa en Feature Engineering.


### Configuración de rutas y utilidades
Define funciones para localizar el proyecto, validar archivos de entrada y manejo seguro de Excel/CSV. Configura las rutas de directorios de datos.

In [1]:
from pathlib import Path
from typing import Iterable, Optional

import pandas as pd

try:
    from IPython.display import display  # Jupyter / VS Code Notebooks
except Exception:
    def display(x):  # fallback minimal
        print(x)

def find_project_root(start: Optional[Path] = None) -> Path:
    """Encuentra la raíz del proyecto sin depender frágilmente del cwd."""
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "01_data_ingestion_enrichment").is_dir() and (p / "02_data_cleaning").is_dir():
            return p
    raise FileNotFoundError(
        "No se pudo localizar la raíz del proyecto.\n"
        "Sugerencia: abre la carpeta raíz del repo en VS Code y vuelve a ejecutar.\n"
        f"Directorio actual (cwd): {start}"
    )

def pick_existing(candidates: Iterable[Path], *, label: str) -> Path:
    candidates = [Path(p) for p in candidates]
    for p in candidates:
        if p.exists():
            return p
    tried = "\n".join([f" - {p.resolve()}" for p in candidates])
    raise FileNotFoundError(f"No se encontró {label}. Rutas probadas:\n{tried}")

def ensure_dir(dir_path: Path) -> Path:
    dir_path = Path(dir_path)
    dir_path.mkdir(parents=True, exist_ok=True)
    return dir_path

def read_excel_checked(path: Path, **kwargs) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo Excel esperado: {path.resolve()}")
    try:
        return pd.read_excel(path, **kwargs)
    except ImportError as e:
        raise ImportError(
            "No puedo leer .xlsx porque falta un motor (normalmente `openpyxl`). "
            "Instala con: pip install openpyxl"
        ) from e

def save_df_csv(
    df: pd.DataFrame,
    out_path: Path,
    *,
    index: bool = False,
    encoding: str = "utf-8-sig",
) -> Path:
    out_path = Path(out_path)
    ensure_dir(out_path.parent)
    df.to_csv(out_path, index=index, encoding=encoding)
    print(f"[OK] Guardado: {out_path.resolve()}")
    print(f"     shape: {df.shape}")
    return out_path

ROOT_DIR = find_project_root()
INGESTION_DIR = ROOT_DIR / "01_data_ingestion_enrichment"
CLEANING_DIR = ROOT_DIR / "02_data_cleaning"
OUTPUT_DIR = ensure_dir(CLEANING_DIR / "outputs")

# Inputs esperados: preferir raíz del proyecto; fallback a 02_data_cleaning/ y 01_data_ingestion_enrichment/
LEADS_FILE = pick_existing(
    [ROOT_DIR / "leads.xlsx", CLEANING_DIR / "leads.xlsx", INGESTION_DIR / "leads.xlsx"],
    label="leads.xlsx (input)",
)
HORAS_FILE = pick_existing(
    [ROOT_DIR / "proyectos_empresa.xlsx", CLEANING_DIR / "proyectos_empresa.xlsx", INGESTION_DIR / "proyectos_empresa.xlsx"],
    label="proyectos_empresa.xlsx (input)",
)

print("ROOT_DIR      ->", ROOT_DIR)
print("INGESTION_DIR ->", INGESTION_DIR)
print("OUTPUT_DIR    ->", OUTPUT_DIR.resolve())
print("LEADS_FILE    ->", LEADS_FILE.resolve())
print("HORAS_FILE    ->", HORAS_FILE.resolve())

ROOT_DIR      -> E:\TESIS MAESTRIA\Desarrollo_clustering_maestria
INGESTION_DIR -> E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\01_data_ingestion_enrichment
OUTPUT_DIR    -> E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs
LEADS_FILE    -> E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\leads.xlsx
HORAS_FILE    -> E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\proyectos_empresa.xlsx


### Carga de datos de entrada
Lee los archivos `leads.xlsx` (Leads) y `proyectos_empresa.xlsx` (Horas) desde la ruta configurada.

In [2]:
# === Carga de datasets (inputs) ===
# - Usa N_FILAS_PRUEBA para iterar rápido (None = dataset completo).
N_FILAS_PRUEBA = None  # p.ej. 500 para prueba rápida

df_leads = read_excel_checked(LEADS_FILE, nrows=N_FILAS_PRUEBA)
df_horas = read_excel_checked(HORAS_FILE, nrows=N_FILAS_PRUEBA)

print("[INPUT] Leads:", df_leads.shape, "|", LEADS_FILE.resolve())
print("[INPUT] Horas:", df_horas.shape, "|", HORAS_FILE.resolve())

[INPUT] Leads: (440, 15) | E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\leads.xlsx
[INPUT] Horas: (412, 18) | E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\proyectos_empresa.xlsx


C:\Users\asus\AppData\Roaming\Python\Python312\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


## Nota de diseño
- Este notebook es **standalone**: lee `leads.xlsx` y `proyectos_empresa.xlsx` desde el proyecto.
- Se encarga exclusivamente de la limpieza y normalización base del nombre comercial, preparando los datos para medir la dificultad del matching automático.
- No depende de dataframes creados por otros notebooks; el intercambio con pasos posteriores es solo vía archivos en `outputs/`.
- Siguiente notebook que consume estos outputs: `02_matching_exacto_scvs.ipynb` (usa `leads_companies_clean.csv` y `horas_empresas_clean.csv`).
- La salida de este carril se conserva como evidencia metodológica y propuesta futura de productización, pero no como fuente operativa del modelo K-Means final.


## 9. Universo unificado de empresas (LEADS ∪ HORAS)

> Objetivo: construir el **universo unificado** de empresas usando los nombres originales normalizados para someterlos al pipeline de cruce automático y documentar la brecha entre nombre comercial y razón social oficial.

- Esta sección exporta los CSV de staging a `02_data_cleaning/outputs/`:
  - `leads_companies_clean.csv`
  - `horas_empresas_clean.csv`
  - `empresas_universe_compilado.csv`


### Limpieza de nombres de LEADS
Normaliza nombres de empresas en Leads de forma estandarizada y genera `leads_companies_clean.csv`.


In [3]:
# === 9.a Limpieza LEADS (Company) ===
# Resultado esperado: DataFrame `leads_companies_clean` con Company raw y normalización final.

import re
import unicodedata
import numpy as np
import pandas as pd


def _strip_accents(text: str) -> str:
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(ch)
    )


def normalize_company_name(value) -> str:
    """Normaliza nombres de empresa para comparación (no para mostrar)."""
    if value is None or pd.isna(value):
        return ""

    s = str(value).strip()
    if not s:
        return ""

    # Quitar rastros de copiado tipo referencias bibliográficas: "1. Nombre [1]".
    s = re.sub(r"\[\s*\d+\s*\]", " ", s)
    s = re.sub(r"^\s*\d+\s*[\.)]\s*", " ", s)

    s = _strip_accents(s)
    s = s.upper()
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)

    # Quitar sufijos legales frecuentes. Ojo: la puntuación ya fue convertida a espacios.
    s = re.sub(
        r"\b(S\s*DE\s*R\s*L\s*DE\s*C\s*V|S\s*R\s*L\s*C\s*V|S\s*A\s*S|SAS|S\s*A|SA|C\s*LTDA|C\s*A|CA|C\s*V|LTDA|CIA|C\s*IA|COMPANIA|COMPAÑIA|ANONIMA|CORP|INC|LLC|C\s*L|\&|Y)\b",
        " ",
        s,
    )
    s = re.sub(r"\b(DE|DEL|LA|EL|LOS|LAS)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    tokens = []
    for tok in s.split():
        if not tokens or tokens[-1] != tok:
            tokens.append(tok)
    return " ".join(tokens)


if "df_leads" not in globals():
    raise NameError("No existe `df_leads` en memoria. Ejecuta primero la celda de 'Carga de datasets'.")

if "Company" not in df_leads.columns:
    raise KeyError("La columna `Company` no existe en `df_leads`.")

_company_raw = (
    df_leads["Company"]
    .astype("string")
    .fillna("")
    .map(lambda x: x.strip())
    .replace("", pd.NA)
    .dropna()
)

leads_companies_clean = pd.DataFrame({"Company_raw": _company_raw})
leads_companies_clean["Company_norm_original"] = leads_companies_clean["Company_raw"].map(normalize_company_name)
leads_companies_clean["Company_match_raw"] = leads_companies_clean["Company_raw"]
leads_companies_clean["Company_norm"] = leads_companies_clean["Company_norm_original"]

leads_companies_clean = (
    leads_companies_clean
    .query("Company_norm != ''")
    .drop_duplicates(subset=["Company_norm"], keep="first")
    .sort_values("Company_norm")
    .reset_index(drop=True)
)

print("[LEADS] Filas originales (no vacías):", int(_company_raw.shape[0]))
print("[LEADS] Empresas distinct (por Company_norm final):", int(leads_companies_clean.shape[0]))

display(leads_companies_clean.head(20))

# --- Export CSV ---
_ = save_df_csv(leads_companies_clean, OUTPUT_DIR / "leads_companies_clean.csv")


[LEADS] Filas originales (no vacías): 440
[LEADS] Empresas distinct (por Company_norm final): 326


,Company_raw,Company_norm_original,Company_match_raw,Company_norm
0,7-ELEVEN MEXICO,7 ELEVEN MEXICO,7-ELEVEN MEXICO,7 ELEVEN MEXICO
1,ABBOTT,ABBOTT,ABBOTT,ABBOTT
2,Abbvie,ABBVIE,Abbvie,ABBVIE
3,ACCO BRANDS,ACCO BRANDS,ACCO BRANDS,ACCO BRANDS
4,"ACH FOOD COMPANIES, INC",ACH FOOD COMPANIES,"ACH FOOD COMPANIES, INC",ACH FOOD COMPANIES
5,ACTINVER,ACTINVER,ACTINVER,ACTINVER
6,ADAMANTINE,ADAMANTINE,ADAMANTINE,ADAMANTINE
7,AFP Genesis,AFP GENESIS,AFP Genesis,AFP GENESIS
8,AIG,AIG,AIG,AIG
9,Akros,AKROS,Akros,AKROS


[OK] Guardado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\leads_companies_clean.csv
     shape: (326, 4)


### Limpieza de HORAS y universo unificado
Normaliza nombres en Horas de forma estandarizada y crea universo de empresas compilado (LEADS ∪ HORAS).


In [4]:
# === 9.b Limpieza HORAS (EMPRESA) ===
# Resultado esperado: DataFrame `horas_empresas_clean` con EMPRESA raw y normalización final.

import pandas as pd

if "df_horas" not in globals():
    raise NameError("No existe `df_horas` en memoria. Ejecuta primero la celda de 'Carga de datasets'.")

if "EMPRESA" not in df_horas.columns:
    raise KeyError("La columna `EMPRESA` no existe en `df_horas`.")

_empresa_raw = (
    df_horas["EMPRESA"]
    .astype("string")
    .fillna("")
    .map(lambda x: x.strip())
    .replace("", pd.NA)
    .dropna()
)

horas_empresas_clean = pd.DataFrame({"EMPRESA_raw": _empresa_raw})
horas_empresas_clean["EMPRESA_norm_original"] = horas_empresas_clean["EMPRESA_raw"].map(normalize_company_name)
horas_empresas_clean["EMPRESA_match_raw"] = horas_empresas_clean["EMPRESA_raw"]
horas_empresas_clean["EMPRESA_norm"] = horas_empresas_clean["EMPRESA_norm_original"]

horas_empresas_clean = (
    horas_empresas_clean
    .query("EMPRESA_norm != ''")
    .drop_duplicates(subset=["EMPRESA_norm"], keep="first")
    .sort_values("EMPRESA_norm")
    .reset_index(drop=True)
)

print("[HORAS] Filas originales (no vacías):", int(_empresa_raw.shape[0]))
print("[HORAS] Empresas distinct (por EMPRESA_norm final):", int(horas_empresas_clean.shape[0]))

display(horas_empresas_clean.head(20))

# --- Export CSV ---
save_df_csv(horas_empresas_clean, OUTPUT_DIR / "horas_empresas_clean.csv")

# --- CSV compilado de empresas (LEADS ∪ HORAS) ---
if "leads_companies_clean" not in globals():
    raise NameError("No existe `leads_companies_clean` en memoria. Ejecuta primero la sección 9.a (Limpieza LEADS).")

def _join_unique(values) -> str:
    vals = []
    for value in values:
        if pd.isna(value):
            continue
        s = str(value).strip()
        if s and s not in vals:
            vals.append(s)
    return " | ".join(vals)

leads_u = pd.DataFrame({
    "source": "LEADS",
    "name_raw": leads_companies_clean["Company_raw"],
    "name_norm": leads_companies_clean["Company_norm"],
    "name_raw_antes_limpieza": leads_companies_clean["Company_raw"],
    "name_norm_antes_limpieza": leads_companies_clean["Company_norm_original"],
    "name_match_raw": leads_companies_clean["Company_match_raw"]
})
horas_u = pd.DataFrame({
    "source": "HORAS",
    "name_raw": horas_empresas_clean["EMPRESA_raw"],
    "name_norm": horas_empresas_clean["EMPRESA_norm"],
    "name_raw_antes_limpieza": horas_empresas_clean["EMPRESA_raw"],
    "name_norm_antes_limpieza": horas_empresas_clean["EMPRESA_norm_original"],
    "name_match_raw": horas_empresas_clean["EMPRESA_match_raw"]
})

universe_all = pd.concat([leads_u, horas_u], ignore_index=True)
universe_distinct = (
    universe_all
    .groupby("name_norm", as_index=False)
    .agg(
        name_raw=("name_raw", "first"),
        sources=("source", lambda s: ",".join(sorted(set(map(str, s))))),
        n_rows=("source", "size"),
        name_norm_antes_limpieza=("name_norm_antes_limpieza", _join_unique),
        name_raw_antes_limpieza=("name_raw_antes_limpieza", _join_unique),
        sources_antes_limpieza=("source", lambda s: ",".join(sorted(set(map(str, s))))),
        n_rows_antes_limpieza=("name_norm_antes_limpieza", "nunique"),
        name_match_raw=("name_match_raw", "first")
    )
    .sort_values("name_norm")
    .reset_index(drop=True)
)

save_df_csv(universe_distinct, OUTPUT_DIR / "empresas_universe_compilado.csv")
display(universe_distinct.head(20))


[HORAS] Filas originales (no vacías): 412
[HORAS] Empresas distinct (por EMPRESA_norm final): 119


,EMPRESA_raw,EMPRESA_norm_original,EMPRESA_match_raw,EMPRESA_norm
0,3dpharma,3DPHARMA,3dpharma,3DPHARMA
1,Adium,ADIUM,Adium,ADIUM
2,Almexa,ALMEXA,Almexa,ALMEXA
3,Alper Seguros,ALPER SEGUROS,Alper Seguros,ALPER SEGUROS
4,Arauco,ARAUCO,Arauco,ARAUCO
5,Aseguradora del Sur,ASEGURADORA SUR,Aseguradora del Sur,ASEGURADORA SUR
6,Asesoría y Control,ASESORIA CONTROL,Asesoría y Control,ASESORIA CONTROL
7,AutoShare,AUTOSHARE,AutoShare,AUTOSHARE
8,AVIS,AVIS,AVIS,AVIS
9,BAC,BAC,BAC,BAC


[OK] Guardado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\horas_empresas_clean.csv
     shape: (119, 4)
[OK] Guardado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\empresas_universe_compilado.csv
     shape: (445, 9)


,name_norm,name_raw,sources,n_rows,name_norm_antes_limpieza,name_raw_antes_limpieza,sources_antes_limpieza,n_rows_antes_limpieza,name_match_raw
0,3DPHARMA,3dpharma,HORAS,1,3DPHARMA,3dpharma,HORAS,1,3dpharma
1,7 ELEVEN MEXICO,7-ELEVEN MEXICO,LEADS,1,7 ELEVEN MEXICO,7-ELEVEN MEXICO,LEADS,1,7-ELEVEN MEXICO
2,ABBOTT,ABBOTT,LEADS,1,ABBOTT,ABBOTT,LEADS,1,ABBOTT
3,ABBVIE,Abbvie,LEADS,1,ABBVIE,Abbvie,LEADS,1,Abbvie
4,ACCO BRANDS,ACCO BRANDS,LEADS,1,ACCO BRANDS,ACCO BRANDS,LEADS,1,ACCO BRANDS
5,ACH FOOD COMPANIES,"ACH FOOD COMPANIES, INC",LEADS,1,ACH FOOD COMPANIES,"ACH FOOD COMPANIES, INC",LEADS,1,"ACH FOOD COMPANIES, INC"
6,ACTINVER,ACTINVER,LEADS,1,ACTINVER,ACTINVER,LEADS,1,ACTINVER
7,ADAMANTINE,ADAMANTINE,LEADS,1,ADAMANTINE,ADAMANTINE,LEADS,1,ADAMANTINE
8,ADIUM,Adium,HORAS,1,ADIUM,Adium,HORAS,1,Adium
9,AFP GENESIS,AFP Genesis,LEADS,1,AFP GENESIS,AFP Genesis,LEADS,1,AFP Genesis


### Verificación de outputs
Comprueba que los tres archivos CSV finales fueron generados correctamente.

In [5]:
# --- Verificación de outputs generados ---
expected_outputs = [
    OUTPUT_DIR / "leads_companies_clean.csv",
    OUTPUT_DIR / "horas_empresas_clean.csv",
    OUTPUT_DIR / "empresas_universe_compilado.csv",
]

print("\n[CHECK] Outputs esperados en:", OUTPUT_DIR.resolve())
missing = []
for p in expected_outputs:
    if p.exists():
        print(" - OK     ", p.resolve())
    else:
        print(" - MISSING", p.resolve())
        missing.append(p)

if missing:
    raise FileNotFoundError(
        "Faltan outputs esperados. Revisa si ejecutaste todas las secciones del notebook.\n"
        + "\n".join([str(p.resolve()) for p in missing])
    )


[CHECK] Outputs esperados en: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs
 - OK      E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\leads_companies_clean.csv
 - OK      E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\horas_empresas_clean.csv
 - OK      E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\empresas_universe_compilado.csv
